### Monthly Reversals

Evidence of Predictable Behavior of Security Returns

In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import clients.finance_client as fin
from statsmodels.tsa.stattools import acf

In [8]:
lseg_df = pd.read_parquet("/Users/dcunning/Code/Python/imperial/SIF/QRT/local_data/data/lseg")

qrt_universe = pd.read_csv('/Users/dcunning/Code/Python/imperial/SIF/QRT/local_data/data/russell_constiuents_2000-2026.csv')['RIC']

lseg_df = lseg_df[lseg_df['RIC'].isin(qrt_universe.unique().tolist()+[".SPX"])]

lseg_df = lseg_df.drop_duplicates(subset=['Date', 'RIC'], keep='last')
df = lseg_df.pivot(index='Date', columns='RIC', values='Close')

In [9]:
df = df[df.index > '2000-01-01']

In [ ]:
# 1. Calculate log returns
df = np.log1p(df)

# 2. Group and sum log returns
monthly_log_rets = (
    df.groupby(['ticker', pd.Grouper(key='Date', freq='ME')])['log_return']
    .sum(min_count=1).reset_index()
)

# 3. Convert back to simple returns
monthly_log_rets['monthly_return'] = np.expm1(monthly_log_rets['log_return'])
monthly_log_rets

/Users/dcunning/Code/Python/imperial/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
df_returns = df.pct_change()

# 2. Create the Lags (1 to 12): 12 shifted wide_df's
# Store them in a dict to easily access during the regression
lags = {f'lag_{i}': df_returns.shift(i) for i in range(1, 13)}

## Rolling 12-Month Regression

1. **Dates**  
   - Estimation: $t$, Target: $t+1$  

2. **Estimation Data**  
   $$
   Y_t = \text{returns at } t, \quad
   X_t = [r_{t-1}, r_{t-2}, \dots, r_{t-12}]
   $$  
   - Keep only stocks with full 13-month history.

3. **OLS Regression**  
$$
   Y_t = \alpha + \sum_{k=1}^{12} \gamma_k r_{t-k} + \epsilon_t
$$  

4. **Forecast**  
   $$
   \hat{Y}_{t+1} = \hat{\alpha} + \sum_{k=1}^{12} \hat{\gamma}_k r_{t-k+1}
   $$

5. **Results**  
   - Store coefficients in `gamma_df`  
   - Store predicted returns in `all_forecasts`

In [16]:
wide_df

ticker,AACB,AACBU,AACG,AAL,AAME,AAOI,AAON,AAPG,AAPL,AARD,...,ZLAB,ZM,ZNB,ZNTL,ZS,ZUMZ,ZURA,ZVRA,ZYBT,ZYME
Date,,,,,,,,,,,,,,,,,,,,,
1990-01-31,NaN,NaN,NaN,NaN,-0.200000,NaN,NaN,NaN,-0.087249,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1990-02-28,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.003222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1990-03-31,NaN,NaN,NaN,NaN,-0.062500,NaN,NaN,NaN,0.183826,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1990-04-30,NaN,NaN,NaN,NaN,0.200000,NaN,NaN,NaN,-0.021738,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1990-05-31,NaN,NaN,NaN,NaN,-0.166667,NaN,NaN,NaN,0.050526,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-30,0.004936,0.003880,-0.364055,-0.159312,-0.148368,0.071488,0.127847,-0.080763,0.096881,0.601205,...,0.023867,0.013265,0.117647,-0.106509,0.081610,0.140780,1.122549e+00,0.048512,-0.720085,0.153275
2025-10-31,0.005108,0.014493,-0.108696,0.168150,-0.013937,0.371385,0.052975,-0.143955,0.061815,-0.195636,...,-0.228976,0.057333,-0.151316,-0.006623,0.105052,0.104029,-1.131640e-01,0.070452,-0.061069,0.117389
2025-11-30,0.003714,0.003810,-0.040650,0.070069,-0.134276,-0.246907,-0.048893,-0.027528,0.032364,-0.066417,...,-0.218905,-0.026023,-0.310077,-0.046667,-0.240503,0.200924,-1.117587e-08,-0.171906,-0.150407,0.399528


In [20]:
results = []
forecast_list = []

for month_idx in range(12, len(wide_df)):
    
    # estimation_date: The 'Today' where we look at current returns to find the patterns
    estimation_date = wide_df.index[month_idx]
    
    # target_date: The 'Tomorrow' we want to trade in
    if month_idx + 1 >= len(wide_df): break
    target_date = wide_df.index[month_idx + 1]
    
    # 2. PREPARE ESTIMATION DATA (Step for finding Gammas)
    # Dependent Variable (Y): All stock returns at the estimation date
    Y_estimation = wide_df.iloc[month_idx] 
    
    # Independent Variables (X): The returns from 1 to 12 months PRIOR to estimation_date
    X_estimation = pd.DataFrame({
        f'lag_{lag}': wide_df.iloc[month_idx - lag] for lag in range(1, 13)
    })
    
    # Keep only stocks that have a full 13-month history at this point
    valid_stocks = pd.concat([Y_estimation, X_estimation], axis=1).dropna()
    
    if len(valid_stocks) < 100: continue # Skip if universe is too small this month
    
    curr_Y = valid_stocks.iloc[:, 0] # y_t
    curr_X = sm.add_constant(valid_stocks.iloc[:, 1:]) # [1, X_t]
    
    # 3. RUN THE REGRESSION
    # We are asking: 'How did the last 12 months explain the returns on estimation_date?'
    model = sm.OLS(curr_Y, curr_X).fit()
    
    # Store the regression coefficients (the 'Gammas') for the Y_t month of 'estimation_date'
    gamma_values = model.params.to_dict()
    gamma_values['Date'] = estimation_date
    results.append(gamma_values)
    
    # 4. FORECAST Y_t+1
    # To predict target_date, 'Lag 1' is actually the return at estimation_date
    X_forecast = pd.DataFrame({
        f'lag_{lag}': wide_df.iloc[month_idx - lag + 1] for lag in range(1, 13)
    })
    X_forecast_with_const = sm.add_constant(X_forecast, has_constant='add')
    
    # Use the fitted 'model' to predict returns at Y_t+1 for 'target_date'
    predictions = model.predict(X_forecast_with_const)
    
    # Store as a DataFrame for easy stacking
    forecast_list.append(
        pd.DataFrame({
        'ticker': predictions.index,
        'pred_return': predictions,
        'Date': target_date  # We label it with the date it belongs to
    }))

# Finalize the data
gamma_df = pd.DataFrame(results).set_index('Date')
all_forecasts = pd.concat(forecast_list).set_index('ticker')

In [21]:
all_forecasts

,pred_return,Date
ticker,,
AACB,NaN,1991-02-28
AACBU,NaN,1991-02-28
AACG,NaN,1991-02-28
AAL,NaN,1991-02-28
AAME,0.038382,1991-02-28
...,...,...
ZUMZ,-0.009982,2026-01-31
ZURA,0.042161,2026-01-31
ZVRA,-0.025560,2026-01-31


In [7]:
# 1. Melt the wide_df back to long format so we can merge it
actual_returns = wide_df.stack().reset_index()
actual_returns.columns = ['Date', 'ticker', 'actual_return']
if 'ticker' in all_forecasts.index.names:
    all_forecasts = all_forecasts.reset_index(drop=True)

# 2. Merge predictions with actuals
# This ensures that for every prediction made for 'Date', we see the 'actual_return' for that same 'Date'
performance_df = pd.merge(all_forecasts, actual_returns, on=['Date', 'ticker'])

# Create deciles (0 to 9) for each date
performance_df['decile'] = performance_df.groupby('Date')['pred_return'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')
)

In [8]:
# 1. Calculate average return per decile per month
monthly_decile_rets = performance_df.groupby(['Date', 'decile'])['actual_return'].mean().unstack()

# 2. Calculate the "Winner-Minus-Loser" (WML) Spread
# This is the return of buying the top decile and shorting the bottom decile
monthly_decile_rets['WML_Spread'] = monthly_decile_rets[9] - monthly_decile_rets[0]

# 3. View the average monthly performance
final_results = monthly_decile_rets.mean()
print(final_results)

t_stat = monthly_decile_rets['WML_Spread'].mean() / (monthly_decile_rets['WML_Spread'].std() / np.sqrt(len(monthly_decile_rets)))
print(f"T-Statistic: {t_stat}")

decile
0             0.040249
1             0.011998
2             0.013300
3             0.022983
4             0.014346
5             0.008175
6             0.026685
7            -0.000164
8             0.024485
9             0.058631
WML_Spread    0.027248
dtype: float32
T-Statistic: 6.243700636079588


In [9]:
sharpe_ratio = (monthly_decile_rets['WML_Spread'].mean() / monthly_decile_rets['WML_Spread'].std()) * np.sqrt(12)
print(f"Annualized Sharpe Ratio: {sharpe_ratio:.2f}")

Annualized Sharpe Ratio: 1.06
